# Notebook 03: Avaliação — Profiling GPU, Métricas e Análise de Gargalos

## Aula 06: Aceleração com Hardware: GPU, TPU e Aceleradores

---

### Objetivos deste Notebook

1. Realizar **profiling detalhado** do modelo usando PyTorch Profiler (Snippet 2)
2. Identificar **gargalos de performance** (operações memory-bound vs compute-bound)
3. Calcular métricas de classificação e avaliar qualidade do modelo
4. Analisar **uso de memória** com e sem gradient checkpointing
5. Estimar **FLOPs e intensidade aritmética** usando Modelo Roofline

### Conexão com o Material Teórico

Este notebook implementa o **Snippet 2** do material da aula:

```python
from torch.profiler import profile, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True
) as prof:
    model(input_tensor)

print(prof.key_averages().table(sort_by='cuda_time_total'))
```

Conforme a seção **'Profiling Avançado: Identificando Gargalos'**:

> "Métricas fundamentais para análise incluem: SM efficiency, memory throughput, 
> e achieved occupancy. Valores de SM efficiency abaixo de 80% tipicamente indicam 
> oportunidades de otimização significativas."

---

In [ ]:
# Imports
import sys
import os
import time
import json
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Adicionar src/ ao path
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_device, set_seed, load_model, format_size, estimate_flops
from src.data_preprocessing import DataPreprocessor
from src.model import SimpleViT, create_model
from src.training import TrainingConfig, Trainer
from src.evaluation import (
    RooflineModel, profile_model, evaluate_model,
    calculate_metrics, benchmark_memory_with_checkpointing,
    plot_confusion_matrix, plot_roofline, plot_training_curves,
)

# Configurações
logging.basicConfig(level=logging.INFO)
set_seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

device = get_device()
print(f"Dispositivo: {device}")

## 1. Carregar Modelo e Dados

Carregamos o modelo treinado no notebook anterior e preparamos o conjunto de teste.

In [ ]:
# Carregar dados
preprocessor = DataPreprocessor(n_samples=10000, img_size=32, n_channels=3, n_classes=10, seed=42)
df = preprocessor.load_data()
df = preprocessor.clean_data(df)
features, labels = preprocessor.prepare_features(df)
splits = preprocessor.split_data(features, labels)

BATCH_SIZE = 64
loaders = preprocessor.create_dataloaders(splits, batch_size=BATCH_SIZE, pin_memory=torch.cuda.is_available())

# Carregar ou criar modelo
model = create_model(embed_dim=128, depth=4, n_heads=4)
model_path = os.path.join('..', 'outputs', 'models', 'model.pt')

try:
    model = load_model(model, model_path, device)
    print("✓ Modelo carregado do checkpoint")
except FileNotFoundError:
    # Se não existir checkpoint, treinar rapidamente
    print("Checkpoint não encontrado. Treinando modelo rapidamente...")
    config = TrainingConfig(n_epochs=3, use_amp=(device.type == 'cuda'), seed=42)
    trainer = Trainer(model, config, device)
    trainer.train(loaders['train'], loaders['val'])
    print("✓ Modelo treinado")

model = model.to(device)
print(f"Parâmetros: {model.count_parameters():,}")

## 2. Avaliação do Modelo — Métricas de Classificação

Avaliamos o modelo no conjunto de teste, medindo acurácia, latência de inferência
e throughput — métricas essenciais para comparação entre aceleradores.

In [ ]:
# Avaliação completa no conjunto de teste
eval_results = evaluate_model(model, loaders['test'], device, n_classes=10)

print("=" * 60)
print("MÉTRICAS DE AVALIAÇÃO")
print("=" * 60)
print(f"  Acurácia geral:          {eval_results['accuracy']:.4f} ({eval_results['n_correct']}/{eval_results['n_total']})")
print(f"  Tempo médio inferência:  {eval_results['avg_inference_time_ms']:.2f} ms/batch")
print(f"  Throughput:              {eval_results['throughput_samples_per_sec']:.0f} amostras/s")
print(f"  Tempo total inferência:  {eval_results['total_inference_time_s']:.3f} s")

print(f"\n  Acurácia por classe:")
for cls, acc in eval_results['per_class_accuracy'].items():
    print(f"    {cls}: {acc:.4f}")

In [ ]:
# Matriz de Confusão
cm = np.array(eval_results['confusion_matrix'])
class_names = [f'Classe {i}' for i in range(10)]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predito', fontsize=12)
ax.set_ylabel('Verdadeiro', fontsize=12)
ax.set_title(f'Matriz de Confusão — SimpleViT (Acurácia: {eval_results["accuracy"]:.4f})', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Profiling com PyTorch Profiler — Snippet 2

Implementa o **Snippet 2** do material da aula para análise detalhada de performance:

```python
from torch.profiler import profile, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True, profile_memory=True
) as prof:
    model(input_tensor)
```

Conforme a seção **'Profiling Avançado: Identificando Gargalos'**:

> "O PyTorch Profiler integra-se nativamente ao framework, capturando traces de 
> execução com overhead mínimo. A integração com TensorBoard permite visualização 
> interativa de flame graphs."

> "Frequentemente, uma única operação ineficiente domina o tempo total, e otimizá-la 
> produz impacto desproporcional no throughput global."

In [ ]:
# Profiling do modelo — Snippet 2 do material da aula
# Captura métricas de CPU e CUDA para identificar gargalos

print("=== PROFILING DO MODELO ===")
print("Executando profiling com PyTorch Profiler...\n")

profile_results = profile_model(
    model, input_shape=(BATCH_SIZE, 3, 32, 32),
    device=device, n_iterations=5, warmup=2,
)

# Tabela de profiling — equivalente a:
# print(prof.key_averages().table(sort_by='cuda_time_total'))
print(profile_results['table'])

In [ ]:
# Análise dos top operadores — identificar gargalos
# Conforme recomendação da seção 'Profiling Avançado':
# "frequentemente, uma única operação ineficiente domina o tempo total"

time_key = 'cuda_time_ms' if device.type == 'cuda' else 'cpu_time_ms'
top_events = profile_results['events'][:10]
total_time = sum(e.get(time_key, 0) for e in top_events)

print("\n=== TOP 10 OPERAÇÕES POR TEMPO ===")
print(f"{'Operação':40s} {'Tempo (ms)':>12s} {'%Total':>8s} {'Count':>6s}")
print("-" * 70)

for evt in top_events:
    t = evt.get(time_key, 0)
    pct = (t / total_time * 100) if total_time > 0 else 0
    print(f"  {evt['name'][:38]:38s} {t:10.3f} ms {pct:7.1f}% {evt['count']:>5d}")

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))

op_names = [e['name'][:30] for e in top_events[:8]]
op_times = [e.get(time_key, 0) for e in top_events[:8]]

# Classificar como compute-bound ou memory-bound
compute_ops = {'addmm', 'mm', 'bmm', 'linear', 'conv2d', 'matmul'}
op_colors = []
for e in top_events[:8]:
    name_lower = e['name'].lower()
    if any(cop in name_lower for cop in compute_ops):
        op_colors.append('#4CAF50')  # Verde = compute-bound
    else:
        op_colors.append('#F44336')  # Vermelho = memory-bound

bars = ax.barh(op_names[::-1], op_times[::-1], color=op_colors[::-1])
ax.set_xlabel(f'Tempo ({time_key.replace("_", " ")})', fontsize=12)
ax.set_title('Top Operações por Tempo de Execução\n(Profiling via PyTorch Profiler)', fontsize=13)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Compute-bound (matmul, linear)'),
    Patch(facecolor='#F44336', label='Memory-bound (softmax, norm, outros)'),
], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/profiling_operacoes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análise de Memória: Gradient Checkpointing

Compara uso de memória GPU com e sem gradient checkpointing,
conforme seção **'Otimização de Memória GPU: Gradient Checkpointing'** (Chen et al., 2016):

> "Checkpointing pode reduzir uso de memória em até 60% em modelos grandes."

Memória convencional escala com O(n) para n camadas; com checkpointing reduz para O(√n).

In [ ]:
# Comparação de memória: com vs sem gradient checkpointing
# Implementa análise da seção 'Gradient Checkpointing' (Chen et al., 2016)

mem_results = benchmark_memory_with_checkpointing(
    model_fn=lambda use_checkpointing: create_model(
        embed_dim=128, depth=4, n_heads=4,
        use_checkpointing=use_checkpointing,
    ),
    input_shape=(BATCH_SIZE, 3, 32, 32),
    device=device,
)

print("\n=== COMPARAÇÃO DE MEMÓRIA: GRADIENT CHECKPOINTING ===")
for config_name, data in mem_results.items():
    if isinstance(data, dict):
        print(f"\n  {config_name}:")
        print(f"    Peak Memory: {data.get('peak_memory_mb', 0):.2f} MB")
        print(f"    Current Memory: {data.get('current_memory_mb', 0):.2f} MB")
        print(f"    Parâmetros: {data.get('n_parameters', 0):,}")
    else:
        print(f"\n  Redução de memória: {data:.1f}%")

In [ ]:
# Visualização de uso de memória
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barras de memória
configs_mem = ['sem_checkpointing', 'com_checkpointing']
peak_mems = [mem_results[c]['peak_memory_mb'] for c in configs_mem if c in mem_results]
labels_mem = ['Sem Checkpointing\n(O(n))', 'Com Checkpointing\n(O(√n))']
colors_mem = ['#F44336', '#4CAF50']

if len(peak_mems) == 2:
    axes[0].bar(labels_mem, peak_mems, color=colors_mem)
    axes[0].set_ylabel('Peak Memory (MB)')
    axes[0].set_title('Peak Memory GPU — Forward + Backward\n(Chen et al., 2016)')
    for i, v in enumerate(peak_mems):
        axes[0].text(i, v * 1.02, f'{v:.1f} MB', ha='center', fontsize=11, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

    # Redução percentual
    reducao = mem_results.get('reducao_pct', 0)
    axes[1].bar(['Redução'], [reducao], color='#4CAF50', width=0.4)
    axes[1].set_ylabel('Redução (%)')
    axes[1].set_title(f'Economia de Memória com Checkpointing')
    axes[1].text(0, reducao * 1.02, f'{reducao:.1f}%', ha='center', fontsize=14, fontweight='bold')
    axes[1].set_ylim(0, max(100, reducao * 1.3))
    axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Gradient Checkpointing: Memória de O(n) → O(√n)\n~33% computação adicional (Chen et al., 2016)',
             fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/memoria_checkpointing.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise de FLOPs e Intensidade Aritmética

Estimamos FLOPs do modelo para classificá-lo no Modelo Roofline (Williams et al., 2009).

$$\text{Intensidade Aritmética} = \frac{\text{FLOPs}}{\text{Bytes Transferidos}}$$

In [ ]:
# Estimar FLOPs do modelo — para análise de roofline
flops_info = estimate_flops(model, input_shape=(BATCH_SIZE, 3, 32, 32))

print("=== ESTIMATIVA DE FLOPs DO MODELO ===")
print(f"  FLOPs estimados: {flops_info['estimated_gflops']:.2f} GFLOPs")
print(f"  Parâmetros:      {flops_info['total_parameters']:,}")
print(f"  Bytes (FP32):    {format_size(flops_info['param_bytes_fp32'])}")
print(f"  Intensidade Aritmética: {flops_info['arithmetic_intensity']:.1f} FLOP/byte")

# Classificar no roofline
roofline_a100 = RooflineModel.from_hardware('A100_80GB')
classification = roofline_a100.classify_operation(flops_info['arithmetic_intensity'])
attainable = roofline_a100.attainable_performance(flops_info['arithmetic_intensity'])

print(f"\n  Classificação Roofline (A100): {classification}")
print(f"  Performance atingível:         {attainable:.1f} TFLOPS")

In [ ]:
# Roofline com o modelo incluído
operations_with_model = dict(RooflineModel.TYPICAL_AI)
operations_with_model['SimpleViT (este modelo)'] = flops_info['arithmetic_intensity']

data = roofline_a100.get_roofline_data()
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(data['arithmetic_intensity'], data['attainable_performance'], 'b-', linewidth=2, label='Roofline A100')
ax.axvline(data['ridge_point'], color='gray', linestyle='--', alpha=0.5,
           label=f"Ridge Point ({data['ridge_point']:.1f})")

colors_cls = {'compute-bound': 'green', 'memory-bound': 'red', 'balanced': 'orange'}
for op_name, ai in operations_with_model.items():
    perf = roofline_a100.attainable_performance(ai)
    cls = roofline_a100.classify_operation(ai)
    marker = '*' if 'SimpleViT' in op_name else 'o'
    size = 15 if 'SimpleViT' in op_name else 8
    ax.plot(ai, perf, marker, color=colors_cls[cls], markersize=size)
    ax.annotate(op_name.replace('_', ' '), (ai, perf),
                textcoords='offset points', xytext=(5, 8), fontsize=8,
                fontweight='bold' if 'SimpleViT' in op_name else 'normal')

ax.set_xlabel('Intensidade Aritmética (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Modelo Roofline — NVIDIA A100: Posição do SimpleViT\n(Williams et al., 2009)', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../outputs/figures/roofline_com_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparação Multi-Hardware (Teórica)

Visualizamos a performance teórica do modelo em diferentes aceleradores,
conforme **Tabela 2** do material da aula:

| Acelerador | Força | Caso de Uso Ideal |
|------------|-------|-------------------|
| GPU (NVIDIA) | Flexibilidade | Treinamento, pesquisa |
| TPU (Google) | Matmul massivo | Treinamento de LLMs |
| NPU (edge) | Eficiência energética | Inferência mobile |

> "A seleção de hardware deve considerar não apenas performance bruta, mas 
> também características do workload específico, custos operacionais, 
> e ecossistema de desenvolvimento." — seção 'Síntese e Integração'

In [ ]:
# Roofline comparativo: múltiplos hardwares
fig, ax = plt.subplots(figsize=(12, 7))

hw_colors = {
    'A100_80GB': '#1976D2', 'H100_SXM': '#0D47A1',
    'V100_32GB': '#43A047', 'RTX_4090': '#E65100',
    'T4': '#7B1FA2', 'CPU_generic': '#757575',
}

for hw_name, color in hw_colors.items():
    roofline_hw = RooflineModel.from_hardware(hw_name)
    data_hw = roofline_hw.get_roofline_data()
    ax.loglog(data_hw['arithmetic_intensity'], data_hw['attainable_performance'],
              '-', linewidth=2, color=color, label=hw_name, alpha=0.8)

# Posicionar nosso modelo
model_ai = flops_info['arithmetic_intensity']
for hw_name in hw_colors:
    roofline_hw = RooflineModel.from_hardware(hw_name)
    perf = roofline_hw.attainable_performance(model_ai)
    ax.plot(model_ai, perf, '*', color=hw_colors[hw_name], markersize=12)

ax.axvline(model_ai, color='gray', linestyle=':', alpha=0.3, label=f'SimpleViT AI={model_ai:.0f}')
ax.set_xlabel('Intensidade Aritmética (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Modelo Roofline Comparativo — Múltiplos Aceleradores\n(Williams et al., 2009; NVIDIA, 2022; Jouppi et al., 2017)',
             fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../outputs/figures/roofline_multi_hardware.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Carregamento de Métricas do Treinamento

Visualizamos as métricas salvas no notebook anterior para análise completa.

In [ ]:
# Carregar métricas do notebook anterior
metrics_path = os.path.join('..', 'outputs', 'logs', 'training_metrics.json')

try:
    with open(metrics_path, 'r') as f:
        training_metrics = json.load(f)
    
    print("✓ Métricas do treinamento carregadas")
    
    # Plotar curvas comparadas
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    styles = {
        'fp32': ('#2196F3', 'FP32 (baseline)'),
        'amp_fp16': ('#4CAF50', 'AMP FP16'),
        'amp_ckpt': ('#FF9800', 'AMP FP16 + Ckpt'),
    }
    
    for key, (color, label) in styles.items():
        if key in training_metrics:
            m = training_metrics[key]
            epochs = range(1, len(m['train_losses']) + 1)
            axes[0].plot(epochs, m['train_losses'], '-o', color=color, label=label, markersize=4)
            if m.get('throughputs'):
                axes[1].plot(epochs, m['throughputs'], '-o', color=color, label=label, markersize=4)
    
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Train Loss')
    axes[0].set_title('Convergência: FP32 vs AMP vs AMP+Ckpt')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Throughput (amostras/s)')
    axes[1].set_title('Throughput por Configuração')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Análise de Performance: Mixed Precision vs Baseline', fontsize=13)
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("⚠ Métricas não encontradas. Execute o notebook 02_treinamento.ipynb primeiro.")

## 8. Pirâmide de Otimização GPU

Conforme a seção **'Síntese e Integração de Conceitos'** do material da aula:

> "A pirâmide de otimização de GPU — algoritmo eficiente, formatos numéricos 
> adequados, gerenciamento de memória, kernel fusion e overlapping de 
> compute/transfer — oferece framework hierárquico para abordar problemas de performance."

In [ ]:
# Visualização da Pirâmide de Otimização GPU
# Figura 2 do material da aula — framework hierárquico

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Pirâmide (de baixo para cima)
levels = [
    (1, 0.5, 8, 1.5, '#1565C0', 'Nível 1: Algoritmo Eficiente\n(Escolha de arquitetura, FlashAttention)'),
    (1.5, 2, 7, 1.5, '#1976D2', 'Nível 2: Formatos Numéricos\n(Mixed Precision: FP16/BF16/TF32)'),
    (2, 3.5, 6, 1.5, '#2196F3', 'Nível 3: Gerenciamento de Memória\n(Gradient Checkpointing, Pin Memory)'),
    (2.5, 5, 5, 1.5, '#42A5F5', 'Nível 4: Kernel Fusion\n(Combinar operações, reduzir overhead)'),
    (3, 6.5, 4, 1.5, '#90CAF9', 'Nível 5: Overlap Compute/Transfer\n(Pipeline CPU↔GPU, prefetch)'),
]

for x, y, w, h, color, text in levels:
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9,
            color='white', fontweight='bold')

ax.set_title('Pirâmide de Otimização de GPU\n(Material da Aula 06 — Síntese e Integração)', fontsize=14)
ax.annotate('Maior\nImpacto', xy=(0.5, 1), fontsize=10, color='#1565C0',
            fontweight='bold', ha='center')
ax.annotate('Menor\nImpacto', xy=(0.5, 8), fontsize=10, color='#90CAF9',
            fontweight='bold', ha='center')

plt.tight_layout()
plt.savefig('../outputs/figures/piramide_otimizacao.png', dpi=150, bbox_inches='tight')
plt.show()

## Resumo

Neste notebook, realizamos:

1. **Avaliação do modelo** — acurácia, latência e throughput de inferência
2. **Profiling com PyTorch Profiler** — identificação de operações dominantes (Snippet 2)
3. **Análise de memória** — comparação com/sem Gradient Checkpointing (Chen et al., 2016)
4. **Estimativa de FLOPs** — posicionamento no Modelo Roofline (Williams et al., 2009)
5. **Comparação multi-hardware** — análise teórica em diferentes aceleradores
6. **Pirâmide de otimização** — framework hierárquico para melhoria de performance

### Principais Conclusões

- Mixed Precision Training mantém qualidade do modelo com speedup significativo em GPU
- Gradient Checkpointing viabiliza modelos maiores em hardware limitado
- O processo de otimização deve seguir a pirâmide: algoritmo → formatos → memória → fusion
- A seleção de hardware depende do workload específico e requisitos operacionais

---

**Voltar para:** [01_exploracao.ipynb](01_exploracao.ipynb) | [02_treinamento.ipynb](02_treinamento.ipynb)